In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
NEWS_API_KEY = "ee512f019a83403da7cb54f9cdae0abb"


StatementMeta(, b2c138cb-e1ff-4971-9070-14c541fc578b, 3, Finished, Available, Finished, False)

In [7]:
import requests

url = "https://newsapi.org/v2/top-headlines"
params = {
    "country": "us",
    "pageSize": 20,
    "apiKey": NEWS_API_KEY
}

StatementMeta(, b2c138cb-e1ff-4971-9070-14c541fc578b, 9, Finished, Available, Finished, False)

In [8]:
response = requests.get(url, params=params)
data = response.json()

print("Status:", data.get("status"))
print("Total results:", data.get("totalResults"))
print("First article title:", data["articles"][0]["title"])

StatementMeta(, b2c138cb-e1ff-4971-9070-14c541fc578b, 10, Finished, Available, Finished, False)

Status: ok
Total results: 35
First article title: Two more cruise ship passengers test positive for hantavirus - Al Jazeera


In [9]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

articles = data["articles"]

# Flatten the nested 'source' field so it's a clean column
rows = [
    Row(
        source_name = a["source"]["name"],
        author      = a.get("author"),
        title       = a.get("title"),
        description = a.get("description"),
        url         = a.get("url"),
        published_at= a.get("publishedAt"),
        content     = a.get("content"),
    )
    for a in articles
]

df = spark.createDataFrame(rows).withColumn("ingested_at", current_timestamp())
df.show(5, truncate=80)

# Save as a Delta table in your lakehouse
df.write.mode("overwrite").saveAsTable("top_headlines")

StatementMeta(, b2c138cb-e1ff-4971-9070-14c541fc578b, 11, Finished, Available, Finished, False)

+------------------+-----------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+--------------------+--------------------------------------------------------------------------------+--------------------------+
|       source_name|           author|                                                                           title|                                                                     description|                                                                             url|        published_at|                                                                         content|               ingested_at|
+------------------+-----------------+--------------------------------------------------------------------------------+-----------------------------------------------------------

In [1]:
spark.sql("SELECT DATE(ingested_at) AS day, COUNT(*) AS total_rows, COUNT(DISTINCT url) AS unique_articles, COUNT(DISTINCT source_name) AS unique_sources FROM top_headlines GROUP BY DATE(ingested_at) ORDER BY day DESC;")

StatementMeta(, b85f070b-496d-4f6a-8334-06a963f1fe55, 3, Finished, Available, Finished, False)

DataFrame[day: date, total_rows: bigint, unique_articles: bigint, unique_sources: bigint]